In [ ]:
import os
import re
import subprocess
import librosa
import shutil
import librosa.display

import soundfile as sf
import noisereduce as nr
import matplotlib.pyplot as plt
import numpy as np

from glob import glob

# Data Extraction & Cleansing

## Getting Script and Audio Files

In [ ]:
def extract_aphasia_clips(cha_file, video_file, output_dir, patient_prefix="patient"):
    """
    Parses a .cha transcript file and extracts corresponding audio clips
    from a source video into 22050Hz mono WAV files.
    """
    # --- 1. SETUP ---
    os.makedirs(output_dir, exist_ok=True)
    
    # Using the AphasiaBank format: "0_5000"
    timestamp_pattern = re.compile(r"(\d+)_(\d+)")
    clip_count = 0
    
    print(f"🎧 Scanning {os.path.basename(cha_file)} and extracting 22050Hz audio...")

    # --- 2. PARSING & EXTRACTION ---
    with open(cha_file, "r", encoding="utf-8") as f:
        for line in f:
            # ONLY look at the Patient's lines
            if line.startswith("*PAR:"):
                match = timestamp_pattern.search(line)
                
                if match:
                    start_ms = int(match.group(1))
                    end_ms = int(match.group(2))
                    
                    # Convert to seconds
                    start_sec = start_ms / 1000.0
                    duration_sec = (end_ms - start_ms) / 1000.0
                    
                    # Clean the transcripts
                    clean_text = line.replace("*PAR:", "").split("")[0].strip()
                    
                    # Skip completely empty lines just in case
                    if not clean_text:
                        continue

                    # Uses the dynamic prefix (e.g., patient1_0000, patient2_0000)
                    base_filename = f"{patient_prefix}_{clip_count:04d}"
                    out_wav = os.path.join(output_dir, f"{base_filename}.wav")
                    out_txt = os.path.join(output_dir, f"{base_filename}.txt")
                    
                    # Save the transcript to a text file
                    with open(out_txt, "w", encoding="utf-8") as txt_file:
                        txt_file.write(clean_text)
                    
                    # Crop and convert the audio using FFmpeg
                    cmd = [
                        "ffmpeg", "-y", 
                        "-ss", str(start_sec), 
                        "-i", video_file, 
                        "-t", str(duration_sec),
                        "-ar", "22050",          # Resample directly to 22050 Hz
                        "-ac", "1",              # Convert to Mono audio
                        "-c:a", "pcm_s16le",     # Uncompressed 16-bit WAV format
                        "-vn",                   # Strip the video track completely
                        out_wav
                    ]
                    
                    # Run FFmpeg silently
                    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                    
                    clip_count += 1
                    
    print(f"✅ Extraction complete! Saved {clip_count} clips for {patient_prefix}.")

In [ ]:
# aprocsa1713a (patient 1)
extract_aphasia_clips(
    cha_file="/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aprocsa1713a.cha",
    video_file="/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aprocsa1713a.mp4",
    output_dir="/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia1",
    patient_prefix="aphasia1"
)

In [ ]:
# aprocsa1731a (patient 2)
extract_aphasia_clips(
    cha_file="/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aprocsa1731a.cha",
    video_file="/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aprocsa1731a.mp4",
    output_dir="/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia2",
    patient_prefix="aphasia2"
)

In [ ]:
# aprocsa1738a (patient 3)
extract_aphasia_clips(
    cha_file="/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aprocsa1738a.cha",
    video_file="/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aprocsa1738a.mp4",
    output_dir="/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia3",
    patient_prefix="aphasia3"
)

In [ ]:
# aprocsa1833a (patient 4)
extract_aphasia_clips(
    cha_file="/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aprocsa1833a.cha",
    video_file="/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aprocsa1833a.mp4",
    output_dir="/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia4",
    patient_prefix="aphasia4"
)

In [ ]:
# aprocsa1944a (patient 5)
extract_aphasia_clips(
    cha_file="/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aprocsa1944a.cha",
    video_file="/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aprocsa1944a.mp4",
    output_dir="/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia5",
    patient_prefix="aphasia5"
)

## Noise Check

In [ ]:
def visualize_stft_spectrogram(audio_path):
    print(f"🎵 Loading audio: {os.path.basename(audio_path)}")
    
    # Load audio with soundfile 
    y, sr = sf.read(audio_path)
    if y.ndim > 1:
        y = y.mean(axis=1) # Convert to mono if it happens to be stereo
        
    # Calculate the STFT
    print("🧮 Calculating Linear STFT...")
    # STFT converts the 1D waveform into a 2D matrix of Frequencies over Time
    stft_matrix = librosa.stft(y)
    stft_magnitude = np.abs(stft_matrix)
    
    # Convert amplitude to decibels (dB) for visualization
    stft_db = librosa.amplitude_to_db(stft_magnitude, ref=np.max)
    
    # Plotting visualization
    fig, ax = plt.subplots(figsize=(10, 6))
    
    img = librosa.display.specshow(
        stft_db, 
        x_axis='time', 
        y_axis='hz', 
        sr=sr, 
        ax=ax, 
        cmap='magma'
    )
    
    ax.set_title(f'Linear STFT: {os.path.basename(audio_path)}', fontsize=16, fontweight='bold')
    fig.colorbar(img, ax=ax, format="%+2.0f dB")
    
    plt.tight_layout()
    plt.show()

    return stft_db, sr

In [ ]:
audio_file = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia1/selected/aphasia1_0132.wav"
stft, sr = visualize_stft_spectrogram(audio_file)

In [ ]:
audio_file = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia2/selected/aphasia2_0017.wav"
stft, sr = visualize_stft_spectrogram(audio_file)

In [ ]:
audio_file = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia3/selected/aphasia3_0156.wav"
stft, sr = visualize_stft_spectrogram(audio_file)

In [ ]:
audio_file = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia4/selected/aphasia4_0179.wav"
stft, sr = visualize_stft_spectrogram(audio_file)

In [ ]:
audio_file = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia5/selected/aphasia5_0234.wav"
stft, sr = visualize_stft_spectrogram(audio_file)

## Noise Cleansing (Single Audio Example)

In [ ]:
def compare_and_clean_noise(good_audio_path, bad_audio_path, output_clean_path):
    print("🧹 Loading audio files...")
    y_good, sr = librosa.load(good_audio_path, sr=22050)
    y_bad, _ = librosa.load(bad_audio_path, sr=22050)
    
    if y_good.ndim > 1: y_good = y_good.mean(axis=1)
    if y_bad.ndim > 1: y_bad = y_bad.mean(axis=1)

    # 1. Grab noise profile
    noise_clip = y_bad[int(3.0 * sr):int(3.5 * sr)]
    
    # 2. Apply Spectral Gating
    print("🧼 Applying Spectral Gating to noisy audio...")
    y_cleaned = nr.reduce_noise(y=y_bad, sr=sr, y_noise=noise_clip, prop_decrease=1.0)
    
    sf.write(output_clean_path, y_cleaned, sr)
    
    # 3. Calculate global dB scale
    print("🚨 Calculating Global DB scaling for fair comparison...")
    stft_good = np.abs(librosa.stft(y_good))
    stft_bad = np.abs(librosa.stft(y_bad))
    stft_cleaned = np.abs(librosa.stft(y_cleaned))

    global_amp_max = max(np.max(stft_good), np.max(stft_bad), np.max(stft_cleaned))
    
    D_good = librosa.amplitude_to_db(stft_good, ref=global_amp_max)
    D_bad = librosa.amplitude_to_db(stft_bad, ref=global_amp_max)
    D_cleaned = librosa.amplitude_to_db(stft_cleaned, ref=global_amp_max)
    
    # 4. Plot the visuals
    fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(20, 6), 
                             gridspec_kw={'width_ratios': [1, 1, 1, 0.05]})
    
    plot_kwargs = {'sr': sr, 'x_axis': 'time', 'y_axis': 'hz', 'cmap': 'magma', 'vmin': -80, 'vmax': 0}
    
    img1 = librosa.display.specshow(D_good, ax=axes[0], **plot_kwargs)
    axes[0].set_title(f'Naturally Clean Audio\n(aprocsa1713a - {os.path.basename(good_audio_path)})', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Frequency (Hz)', fontsize=12)
    axes[0].set_xlabel('Time (Seconds)', fontsize=12)

    img2 = librosa.display.specshow(D_bad, ax=axes[1], **plot_kwargs)
    axes[1].set_title(f'Original Noisy Audio\n(aprocsa1731a - {os.path.basename(bad_audio_path)})', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Time (Seconds)', fontsize=12)
    axes[1].set_ylabel('')
    axes[1].set_yticklabels([])

    img3 = librosa.display.specshow(D_cleaned, ax=axes[2], **plot_kwargs)
    axes[2].set_title(f'Spectral Gating Noise Reduction\n(aprocsa1731a - {os.path.basename(bad_audio_path)})', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('Time (Seconds)', fontsize=12)
    axes[2].set_ylabel('') 
    axes[2].set_yticklabels([]) 
    
    for i in range(3):
        axes[i].tick_params(axis='both', which='major', labelsize=11)
        axes[i].set_ylim(0, sr/2) 
    
    cbar = fig.colorbar(img3, cax=axes[3])
    cbar.set_label('Acoustic Amplitude (dB)', rotation=270, labelpad=15, fontsize=12)
    cbar.ax.tick_params(labelsize=11)
    
    plt.tight_layout()
    plt.show()
    
    return y_cleaned, sr

In [ ]:
GOOD_AUDIO = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia1/selected/aphasia1_0132.wav"
BAD_AUDIO = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia2/selected/aphasia2_0017.wav"
OUTPUT_CLEAN = "/rds/general/user/ak8224/home/MedEmoji-TTS/jupyter/dataset_analysis/cleaned_example.wav"

compare_and_clean_noise(GOOD_AUDIO, BAD_AUDIO, OUTPUT_CLEAN)

## Batch Noise Cleansing

In [ ]:
def batch_clean_folder(reference_audio_path, input_folder, output_folder):
    print(f"🔍 Extracting noise profile from: {reference_audio_path}...")
    
    # 1. Load the reference audio to get the exact mic hiss
    y_ref, sr = librosa.load(reference_audio_path, sr=22050)
    
    # Grab the Noise Profile (Seconds x to y)
    noise_clip = y_ref[int(1.2 * sr):int(1.7 * sr)]
    
    # 2. Ensure output folder exists
    os.makedirs(output_folder, exist_ok=True)
    
    # 3. Find all .wav files in the input folder
    audio_files = glob(os.path.join(input_folder, "*.wav"))
    print(f"📁 Found {len(audio_files)} files to clean. Starting batch process...\n")
    
    # 4. Loop through and clean every file
    for file_path in audio_files:
        filename = os.path.basename(file_path)
        out_path = os.path.join(output_folder, filename)
        
        y_noisy, _ = librosa.load(file_path, sr=sr)
        
        y_clean = nr.reduce_noise(y=y_noisy, sr=sr, y_noise=noise_clip, prop_decrease=1.0)
        
        sf.write(out_path, y_clean, sr)
        
    print("\n🎉 Batch cleaning complete!")

In [ ]:
# Bad reference timing: 3.0 - 3.5 seconds
REFERENCE_FILE = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia2/selected/aphasia2_0017.wav" 
INPUT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia2/selected"
OUTPUT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia2/filtered"

batch_clean_folder(REFERENCE_FILE, INPUT_DIR, OUTPUT_DIR)

In [ ]:
# Bad reference timing: 1.2 - 1.7 seconds
REFERENCE_FILE = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia4/selected/aphasia4_0179.wav" 
INPUT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia4/selected"
OUTPUT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia4/filtered"

batch_clean_folder(REFERENCE_FILE, INPUT_DIR, OUTPUT_DIR)

# Calculating Metrics

## Speech-to-Pause Ratio

In [ ]:
def analyze_single_vad(audio_path, top_db=15):
    # 1. Audio Load with soundfile
    y, sr = sf.read(audio_path)
    
    if y.ndim > 1:
        y = y.mean(axis=1)
        
    total_duration = len(y) / sr
    
    # 2. Run the VAD with 15dB threshold
    speech_intervals = librosa.effects.split(y, top_db=top_db)
    speech_samples = sum([end - start for start, end in speech_intervals])
    
    speech_duration = speech_samples / sr
    pause_duration = total_duration - speech_duration
    
    # 3. Calculate metrics
    speech_ratio = speech_duration / total_duration if total_duration > 0 else 0
    pause_ratio = pause_duration / total_duration if total_duration > 0 else 0
    
    return {
        "filename": os.path.basename(audio_path),
        "total_sec": total_duration,
        "speech_sec": speech_duration,
        "pause_sec": pause_duration,
        "speech_ratio": float(f"{speech_ratio:.4f}"),
        "pause_ratio": float(f"{pause_ratio:.4f}")
    }

def batch_analyze_directory_vad(directory_path):
    audio_files = glob(os.path.join(directory_path, "*.wav"))
    
    if not audio_files:
        print(f"❌ No .wav files found in {directory_path}")
        return
        
    print(f"🔍 Analyzing {len(audio_files)} files in directory: {directory_path}\n")
    
    total_dataset_duration = 0.0
    total_dataset_speech = 0.0
    total_dataset_pause = 0.0
    
    # Process each file and print individual stats
    for file_path in audio_files:
        stats = analyze_single_vad(file_path, top_db=15)
        
        # Aggregate global metrics
        total_dataset_duration += stats["total_sec"]
        total_dataset_speech += stats["speech_sec"]
        total_dataset_pause += stats["pause_sec"]
    
    # Calculate Global Ratios
    global_speech_ratio = total_dataset_speech / total_dataset_duration
    global_pause_ratio = total_dataset_pause / total_dataset_duration
    
    print("\n" + "="*50)
    print("📊 GLOBAL DATASET VAD BASELINE")
    print("="*50)
    print(f"Total Audio Processed : {total_dataset_duration / 60:.2f} minutes")
    print(f"Global Speech Ratio   : {global_speech_ratio:.4f}")
    print(f"Global Pause Ratio    : {global_pause_ratio:.4f}")
    print("="*50)

In [ ]:
# aprocsa1713a (patient 1)
TARGET_DIRECTORY = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia1/selected"
batch_analyze_directory_vad(TARGET_DIRECTORY)

In [ ]:
# aprocsa1731a (patient 2)
TARGET_DIRECTORY = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia2/filtered"
batch_analyze_directory_vad(TARGET_DIRECTORY)

In [ ]:
# aprocsa1738a (patient 3)
TARGET_DIRECTORY = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia3/selected"
batch_analyze_directory_vad(TARGET_DIRECTORY)

In [ ]:
# aprocsa1833a (patient 4)
TARGET_DIRECTORY = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia4/filtered"
batch_analyze_directory_vad(TARGET_DIRECTORY)

In [ ]:
# aprocsa1944a (patient 5)
TARGET_DIRECTORY = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia5/selected"
batch_analyze_directory_vad(TARGET_DIRECTORY)

## Disfluencies ("uhm" "uh" "um") and Formulaic Fillers ("you know")

In [ ]:
def normalize_disfluency_metrics(audio_dir, script_dir, new_script_dir):
    # 1. Create the new directory for the curated scripts
    os.makedirs(new_script_dir, exist_ok=True)
    
    # Grab only the manually curated audio files
    audio_files = glob(os.path.join(audio_dir, "*.wav"))
    
    if not audio_files:
        print(f"❌ No .wav files found in {audio_dir}")
        return
        
    print(f"🔍 Found {len(audio_files)} curated audio clips. Starting analysis...\n")
    
    filler_pattern = re.compile(r'\b(um|uh|uhm)\b', re.IGNORECASE)
    phrase_pattern = re.compile(r'\b(I don\'t know|I mean|let\'s see|you_know)\b', re.IGNORECASE)
    
    total_fillers = 0
    total_you_knows = 0
    total_duration_sec = 0.0
    files_processed = 0
    
    for audio_path in audio_files:
        base_name = os.path.basename(audio_path).replace(".wav", ".txt")
        old_txt_path = os.path.join(script_dir, base_name)
        new_txt_path = os.path.join(new_script_dir, base_name)
        
        # 1. Check if the script exists in the old directory and move it
        if os.path.exists(old_txt_path):
            shutil.move(old_txt_path, new_txt_path)
            target_txt = new_txt_path
        elif os.path.exists(new_txt_path):
            target_txt = new_txt_path
        else:
            print(f"⚠️ Warning: No matching transcript found for {base_name}")
            continue
            
        # 2. Get the exact audio duration instantly using soundfile.info
        duration = sf.info(audio_path).duration
        total_duration_sec += duration
        
        # 3. Read the text and count the disfluencies
        with open(target_txt, 'r', encoding='utf-8') as f_txt:
            text = f_txt.read()
            
        total_fillers += len(filler_pattern.findall(text))
        total_you_knows += len(phrase_pattern.findall(text))
        files_processed += 1
        
    # 4. Calculate the Per-Minute Rates
    total_duration_min = total_duration_sec / 60.0
    
    fillers_per_min = total_fillers / total_duration_min if total_duration_min > 0 else 0.0
    you_knows_per_min = total_you_knows / total_duration_min if total_duration_min > 0 else 0.0
    
    print("="*50)
    print("📊 CURATED DATASET DISFLUENCY METRICS")
    print("="*50)
    print(f"Valid Files Processed                               : {files_processed}")
    print(f"Total Audio Duration                                : {total_duration_min:.4f} minutes")
    print(f"Total 'Um/Uh/Uhm'                                   : {total_fillers}")
    print(f"Total 'You know/I don't know/I mean/Let's see'      : {total_you_knows}")
    print("-" * 50)
    print(f"Fillers Rate                                        : {fillers_per_min:.4f} per minute")
    print(f"Formulaic Phrase Rate                               : {you_knows_per_min:.4f} per minute")
    print("="*50)

In [ ]:
# aprocsa1713a (patient 1)
AUDIO_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia1/selected"
OLD_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia1/scripts"
NEW_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia1/filtered_scripts"

normalize_disfluency_metrics(AUDIO_DIR, OLD_SCRIPT_DIR, NEW_SCRIPT_DIR)

In [ ]:
# aprocsa1731a (patient 2)
AUDIO_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia2/filtered"
OLD_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia2/scripts"
NEW_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia2/filtered_scripts"

normalize_disfluency_metrics(AUDIO_DIR, OLD_SCRIPT_DIR, NEW_SCRIPT_DIR)

In [ ]:
# aprocsa1738a (patient 3)
AUDIO_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia3/selected"
OLD_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia3/scripts"
NEW_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia3/filtered_scripts"

normalize_disfluency_metrics(AUDIO_DIR, OLD_SCRIPT_DIR, NEW_SCRIPT_DIR)

In [ ]:
# aprocsa1833a (patient 4)
AUDIO_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia4/filtered"
OLD_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia4/scripts"
NEW_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia4/filtered_scripts"

normalize_disfluency_metrics(AUDIO_DIR, OLD_SCRIPT_DIR, NEW_SCRIPT_DIR)

In [ ]:
# aprocsa1944a (patient 5)
AUDIO_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia5/selected"
OLD_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia5/scripts"
NEW_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia5/filtered_scripts"

normalize_disfluency_metrics(AUDIO_DIR, OLD_SCRIPT_DIR, NEW_SCRIPT_DIR)

In [ ]:
# aprocsa1944a (patient 5 - batch 2)
AUDIO_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia5/selected2"
OLD_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia5/scripts"
NEW_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia5/filtered_scripts2"

normalize_disfluency_metrics(AUDIO_DIR, OLD_SCRIPT_DIR, NEW_SCRIPT_DIR)

In [ ]:
# aprocsa1944a (patient 3 - batch 2)
AUDIO_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia3/selected2"
OLD_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia3/scripts"
NEW_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia3/filtered_scripts2"

normalize_disfluency_metrics(AUDIO_DIR, OLD_SCRIPT_DIR, NEW_SCRIPT_DIR)

## MFCC Mean and Variance

In [ ]:
def analyze_single_mfcc_split(audio_path, top_db=15):
    y, sr = sf.read(audio_path)
    if y.ndim > 1:
        y = y.mean(axis=1)
        
    # 1. Strip the silence! 
    speech_intervals = librosa.effects.split(y, top_db=top_db)
    
    if len(speech_intervals) == 0:
        return None 
        
    speech_y = np.concatenate([y[start:end] for start, end in speech_intervals])
    
    # 2. Extract MFCCs
    mfccs = librosa.feature.mfcc(y=speech_y, sr=sr, n_mfcc=13)
    
    # 3. Drop the 0th coefficient (energy/volume)
    mfccs = mfccs[1:, :]
    
    # 4. Calculate the Standard Deviation for EACH of the 12 coefficients
    coeff_stds = np.std(mfccs, axis=1)
    
    # 5. Splitting the Analysis
    lower_tract_std = np.mean(coeff_stds[0:4])
    upper_tract_std = np.mean(coeff_stds[4:])
    
    return {
        "filename": os.path.basename(audio_path),
        "mfcc_matrix": mfccs, 
        "all_stds": coeff_stds.tolist(),
        "lower_tract_std": float(f"{lower_tract_std:.4f}"),
        "upper_tract_std": float(f"{upper_tract_std:.4f}"),
        "overall_std": float(f"{np.mean(coeff_stds):.4f}")
    }

def batch_analyze_directory_mfcc(directory_path, top_db=15):
    audio_files = glob(os.path.join(directory_path, "*.wav"))
    
    if not audio_files:
        print(f"❌ No .wav files found in {directory_path}")
        return None
    
    global_mfcc_list = []
    
    for file_path in audio_files:
        stats = analyze_single_mfcc_split(file_path, top_db) 
        
        if stats is not None:
            global_mfcc_list.append(stats["mfcc_matrix"])
            
    if not global_mfcc_list:
        print("No active speech found in dataset.")
        return None
        
    global_mfcc_matrix = np.concatenate(global_mfcc_list, axis=1)
    
    # Calculate Global Stats for all 12 coefficients
    global_coeff_stds = np.std(global_mfcc_matrix, axis=1)
    global_std = np.mean(global_coeff_stds)
    
    global_coeff_means = np.mean(global_mfcc_matrix, axis=1)
    global_mean = np.mean(global_coeff_means)
    
    global_lower_tract_mean = np.mean(global_coeff_means[0:4])
    global_lower_tract_std = np.mean(global_coeff_stds[0:4])
    
    global_upper_tract_mean = np.mean(global_coeff_means[4:])
    global_upper_tract_std = np.mean(global_coeff_stds[4:])
    
    print("\n" + "="*50)
    print(f"📊 GLOBAL MFCC BASELINE: {os.path.basename(os.path.normpath(directory_path))}")
    print("="*50)
    print(f"Total Active Speech Frames : {global_mfcc_matrix.shape[1]}")
    print(f"Global MFCC Mean           : {global_mean:.4f}")
    print(f"Global Overall Std Dev     : {global_std:.4f}")
    print("-" * 50)
    print(f"Lower Tract Mean (1-4)     : {global_lower_tract_mean:.4f}")
    print(f"Lower Tract Std Dev (1-4)  : {global_lower_tract_std:.4f}")
    print(f"Upper Tract Mean (5-12)    : {global_upper_tract_mean:.4f}")
    print(f"Upper Tract Std Dev (5-12) : {global_upper_tract_std:.4f}")
    print("="*50)

    return {
        "global_matrix": global_mfcc_matrix,
    }

In [ ]:
# aprocsa1713a (patient 1)
TARGET_DIRECTORY = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia1/selected"
global_mfcc_aphasia = batch_analyze_directory_mfcc(TARGET_DIRECTORY)

In [ ]:
# aprocsa1731a (patient 2)
TARGET_DIRECTORY = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia2/filtered"
global_mfcc_aphasia = batch_analyze_directory_mfcc(TARGET_DIRECTORY)

In [ ]:
# aprocsa1738a (patient 3)
TARGET_DIRECTORY = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia3/selected"
global_mfcc_aphasia = batch_analyze_directory_mfcc(TARGET_DIRECTORY)

In [ ]:
# aprocsa1833a (patient 4)
TARGET_DIRECTORY = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia4/filtered"
global_mfcc_aphasia = batch_analyze_directory_mfcc(TARGET_DIRECTORY)

In [ ]:
# aprocsa1944a (patient 5)
TARGET_DIRECTORY = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia5/selected"
global_mfcc_aphasia = batch_analyze_directory_mfcc(TARGET_DIRECTORY)

## Word per Minutes (WPM)

In [ ]:
def calculate_directory_wpm(txt_dir, total_audio_duration_minutes):
    txt_files = glob(os.path.join(txt_dir, "*.txt"))
    
    if not txt_files:
        print(f"❌ No .txt files found in {txt_dir}")
        return 0, 0
        
    print(f"🧮 Calculating Effective WPM across {len(txt_files)} curated transcripts...")
    
    fillers_to_ignore = {'um', 'uh', 'uhm'}
    total_effective_words = 0
    
    for file_path in txt_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()
            
        # 1. Strip gesture/vocal tags (e.g., '&=ges' is deleted, leaving 'excited')
        text = re.sub(r'&=\S+', '', text)
        
        # 2. Delete abandoned phrases and repetitions
        text = re.sub(r'<[^>]+>\s*\[//?\]', '', text) 
        
        # 3. Clean up any remaining formatting brackets and pauses
        text = re.sub(r'\[.*?\]', '', text) 
        text = re.sub(r'[<>]', '', text) 
        text = re.sub(r'\(\.{1,3}\)', '', text)
        
        # 4. Strip prefixes like &- and &+ so '&-um' becomes 'um'
        text = re.sub(r'&[-+]', '', text)
        
        # 5. Remove punctuation and split into raw words
        clean_text = re.sub(r'[^\w\s]', '', text.lower())
        words = clean_text.split()
        
        # 6. Apply the final word-by-word logic
        effective_words = []
        for word in words:
            # Skip the classic fillers
            if word in fillers_to_ignore:
                continue
                
            # Skip consecutive immediate duplicates
            if effective_words and word == effective_words[-1]:
                continue
                
            effective_words.append(word)
            
        total_effective_words += len(effective_words)
        
    # Calculate the final global WPM
    wpm = total_effective_words / total_audio_duration_minutes if total_audio_duration_minutes > 0 else 0
    
    print("\n" + "="*50)
    print("📊 EFFECTIVE WORDS PER MINUTE (WPM) BASELINE")
    print("="*50)
    print(f"Total Curated Files   : {len(txt_files)}")
    print(f"Total Audio Duration  : {total_audio_duration_minutes:.4f} minutes")
    print(f"Total Effective Words : {total_effective_words}")
    print("-" * 50)
    print(f"Effective WPM         : {wpm:.4f} words/min")
    print("="*50)
    
    return wpm, total_effective_words

In [ ]:
# aprocsa1713a (patient 1)
NEW_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia1/filtered_scripts"

wpm, words = calculate_directory_wpm(NEW_SCRIPT_DIR, total_audio_duration_minutes=7.3924)

In [ ]:
# aprocsa1731a (patient 2)
NEW_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia2/filtered_scripts"

wpm, words = calculate_directory_wpm(NEW_SCRIPT_DIR, total_audio_duration_minutes=3.9424)

In [ ]:
# aprocsa1738a (patient 3)
NEW_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia3/filtered_scripts"

wpm, words = calculate_directory_wpm(NEW_SCRIPT_DIR, total_audio_duration_minutes=7.2632)

In [ ]:
# aprocsa1833a (patient 4)
NEW_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia4/filtered_scripts"

wpm, words = calculate_directory_wpm(NEW_SCRIPT_DIR, total_audio_duration_minutes=9.3145)

In [ ]:
# aprocsa1944a (patient 5)
NEW_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia5/filtered_scripts"

wpm, words = calculate_directory_wpm(NEW_SCRIPT_DIR, total_audio_duration_minutes=13.1688)